In [1]:
import pandas as pd

df = pd.read_csv("cleaned_transactions.csv")

df["transaction_date"] = pd.to_datetime(df["transaction_date"])

In [2]:
df["exceeds_balance"] = 0  # start everyone at 0 (default: does not exceed)

debit_mask = (df["transaction_direction"] == "Debit") & (df["transaction_amount"] > df["account_balance"])
df.loc[debit_mask, "exceeds_balance"] = 1

print("Rows flagged as exceeds_balance:", df["exceeds_balance"].sum())

Rows flagged as exceeds_balance: 43902


In [3]:
df["amount_to_balance_ratio"] = df["transaction_amount"] / df["account_balance"]

# quick look at the result
df["amount_to_balance_ratio"].describe()

count    550000.000000
mean          1.914231
std          19.872191
min           0.000012
25%           0.015683
50%           0.065597
75%           0.338589
max        6778.062322
Name: amount_to_balance_ratio, dtype: float64

In [4]:
df["amount_to_balance_ratio"] = df["amount_to_balance_ratio"].clip(upper=10)

In [5]:
df["amount_to_balance_ratio"].describe()

count    550000.000000
mean          0.795525
std           2.088852
min           0.000012
25%           0.015683
50%           0.065597
75%           0.338589
max          10.000000
Name: amount_to_balance_ratio, dtype: float64

In [8]:
df = df.sort_values(by=["customer_id", "transaction_date"])

df["customer_avg_amount"] = (
    df.groupby("customer_id")["transaction_amount"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [9]:
df[["customer_id", "transaction_date", "transaction_amount", "customer_avg_amount"]].head(20)

,customer_id,transaction_date,transaction_amount,customer_avg_amount
136915,CUST000001,2020-03-30,892.86,NaN
257545,CUST000001,2021-05-06,328.91,892.860000
283984,CUST000001,2021-08-02,1000.00,610.885000
453096,CUST000001,2023-02-13,584.18,740.590000
545069,CUST000001,2023-12-16,470.63,701.487500
60043,CUST000002,2019-07-19,65982.76,NaN
100722,CUST000002,2019-11-30,186.60,65982.760000
150435,CUST000002,2020-05-15,14880.67,33084.680000
185159,CUST000002,2020-09-07,384.28,27016.676667
239471,CUST000002,2021-03-06,1783.51,20358.577500


In [10]:
df["amount_vs_customer_avg"] = df["transaction_amount"] / df["customer_avg_amount"]
df["amount_vs_customer_avg"].describe()

count    470084.000000
mean         14.211842
std         172.997135
min           0.000020
25%           0.065991
50%           0.351840
75%           2.003143
max       36597.626947
Name: amount_vs_customer_avg, dtype: float64

In [11]:
df["amount_vs_customer_avg"] = df["amount_vs_customer_avg"].fillna(1.0)

In [12]:
df["amount_vs_customer_avg"] = df["amount_vs_customer_avg"].clip(upper=10)

In [13]:
df["amount_vs_customer_avg"].describe()

count    550000.000000
mean          1.869479
std           3.055634
min           0.000020
25%           0.090660
50%           0.603080
75%           1.418542
max          10.000000
Name: amount_vs_customer_avg, dtype: float64

In [14]:
df.groupby("is_fraud")["amount_vs_customer_avg"].mean()

is_fraud
0    1.864304
1    2.448466
Name: amount_vs_customer_avg, dtype: float64

In [15]:
df.groupby("is_fraud")["amount_to_balance_ratio"].mean()

is_fraud
0    0.788882
1    1.538583
Name: amount_to_balance_ratio, dtype: float64

In [16]:
df["customer_transaction_count"] = df.groupby("customer_id").cumcount() + 1

In [17]:
df[["customer_id", "transaction_date", "customer_transaction_count"]].head(10)

,customer_id,transaction_date,customer_transaction_count
136915,CUST000001,2020-03-30,1
257545,CUST000001,2021-05-06,2
283984,CUST000001,2021-08-02,3
453096,CUST000001,2023-02-13,4
545069,CUST000001,2023-12-16,5
60043,CUST000002,2019-07-19,1
100722,CUST000002,2019-11-30,2
150435,CUST000002,2020-05-15,3
185159,CUST000002,2020-09-07,4
239471,CUST000002,2021-03-06,5


In [18]:
df["days_since_last_transaction"] = (
    df["transaction_date"] - df.groupby("customer_id")["transaction_date"].shift(1)
).dt.days

In [19]:
df[["customer_id", "transaction_date", "days_since_last_transaction"]].head(10)

,customer_id,transaction_date,days_since_last_transaction
136915,CUST000001,2020-03-30,NaN
257545,CUST000001,2021-05-06,402.0
283984,CUST000001,2021-08-02,88.0
453096,CUST000001,2023-02-13,560.0
545069,CUST000001,2023-12-16,306.0
60043,CUST000002,2019-07-19,NaN
100722,CUST000002,2019-11-30,134.0
150435,CUST000002,2020-05-15,167.0
185159,CUST000002,2020-09-07,115.0
239471,CUST000002,2021-03-06,180.0


In [20]:
df.groupby("is_fraud")["days_since_last_transaction"].mean()

is_fraud
0    220.810236
1    215.076960
Name: days_since_last_transaction, dtype: float64

In [22]:
median_gap = df["days_since_last_transaction"].median()

df["days_since_last_transaction"] = df["days_since_last_transaction"].fillna(median_gap)

print("Filled with:", median_gap)

Filled with: 156.0


In [23]:
df.to_csv("feature_engineered_transactions.csv", index=False)
print("Saved:", df.shape)

Saved: (550000, 26)
